# Aula 6 — pandas III e NumPy
**Operações vetorizadas, variáveis derivadas e agregações**

Fund. de Programação, Dados e Estatística para IA · Einstein

Objetivo: transformar a base preparada na Aula 5 em uma base analiticamente enriquecida.

## 01. Ler e conferir

In [23]:
import pandas as pd
import numpy as np

Se o arquivo `pacientes_aula6_limpo.csv` não estiver no ambiente, a célula abaixo cria uma base de demonstração com a mesma estrutura.

In [24]:
df = pd.read_csv("pacientes_aula6_limpo.csv", parse_dates=["data_nascimento", "data_admissao", "data_alta"])
df.head()

,id_paciente,data_nascimento,idade,data_admissao,data_alta,tempo_internacao_dias,internacoes_anteriores,servico,readmissao_30d
0,101,1944-08-21,81,2026-03-01,2026-03-13,12.0,0,Cardiologia,0
1,102,1992-02-15,34,2026-03-22,2026-03-30,8.0,0,Clínica Médica,0
2,103,1984-01-15,42,2026-05-14,2026-05-19,5.0,0,Neurologia,1
3,104,1978-09-22,47,2026-03-23,2026-03-24,1.0,1,Ortopedia,0
4,105,1938-10-14,87,2026-04-11,2026-04-16,5.0,2,Cardiologia,0


In [25]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

(60, 9)
id_paciente                        int64
data_nascimento           datetime64[ns]
idade                              int64
data_admissao             datetime64[ns]
data_alta                 datetime64[ns]
tempo_internacao_dias            float64
internacoes_anteriores             int64
servico                           object
readmissao_30d                     int64
dtype: object
id_paciente               0
data_nascimento           0
idade                     0
data_admissao             0
data_alta                 0
tempo_internacao_dias     2
internacoes_anteriores    0
servico                   0
readmissao_30d            0
dtype: int64


## 02. Operar sobre colunas

In [44]:
# Com laço
horas_loop = []
for dias in df["tempo_internacao_dias"]:
    horas_loop.append(dias * 24)

df["horas_loop"] = horas_loop
print(df["tempo_internacao_dias"].head(), df["horas_loop"].head())

0    12.0
1     8.0
2     5.0
3     1.0
4     5.0
Name: tempo_internacao_dias, dtype: float64 0    288.0
1    192.0
2    120.0
3     24.0
4    120.0
Name: horas_loop, dtype: float64


In [52]:
# Vetorizada
df["tempo_internacao_horas"] = (
    df["tempo_internacao_dias"] * 24
)
print(df["tempo_internacao_dias"].head(), df["tempo_internacao_horas"].head())

0    12.0
1     8.0
2     5.0
3     1.0
4     5.0
Name: tempo_internacao_dias, dtype: float64 0    288.0
1    192.0
2    120.0
3     24.0
4    120.0
Name: tempo_internacao_horas, dtype: float64


Conferindo se as duas versões produzem o mesmo resultado:

In [46]:
np.allclose(
    df["horas_loop"],
    df["tempo_internacao_horas"],
    equal_nan=True
)

True

## 03. Criar variáveis
### Idade a partir de datas

In [55]:
idade_dias = (df["data_admissao"] - df["data_nascimento"]).dt.days
df["idade_calculada"] = np.floor(idade_dias / 365.25).astype("Int64")

df[["idade", "idade_calculada"]].head()

,idade,idade_calculada
0,81,81
1,34,34
2,42,42
3,47,47
4,87,87


### Comparar o tempo informado com o calculado
Antes de corrigir, entender a regra de cada coluna: arredondamento, horas parciais ou outra definição.

In [58]:
df["tempo_calculado"] = (df["data_alta"] - df["data_admissao"]).dt.days
df["diferenca_tempo"] = df["tempo_internacao_dias"] - df["tempo_calculado"]

df.loc[df["diferenca_tempo"] != 0, ["id_paciente", "diferenca_tempo"]]

,id_paciente,diferenca_tempo
5,106,1.0
9,110,NaN
18,119,1.0
33,134,NaN
41,142,1.0


Note que linhas com `tempo_internacao_dias` ausente também aparecem: `NaN != 0` é verdadeiro.

## 04. Classificar com condições
### np.where(): duas categorias
O limite de 7 dias é didático. Limiares em saúde vêm de protocolo, literatura ou desenho do estudo.

In [62]:
df["internacao_prolongada"] = np.where(
    df["tempo_internacao_dias"] > 7, "sim", "nao"
)

df["internacao_prolongada"].value_counts()

,count
internacao_prolongada,
nao,36
sim,24


### np.select(): várias categorias
A primeira condição verdadeira define a categoria; `default` mostra o que ficou sem classificação.

In [63]:
condicoes = [
    df["internacoes_anteriores"] == 0,
    df["internacoes_anteriores"].between(1, 2),
    df["internacoes_anteriores"] >= 3
]
categorias = ["nenhuma", "1 a 2", "3 ou mais"]

df["historico"] = np.select(
    condicoes, categorias, default="nao classificado"
)

df["historico"].value_counts()

,count
historico,
nenhuma,26
1 a 2,20
3 ou mais,14


> Variáveis derivadas precisam de significado, definição e propósito. Uma fórmula sem justificativa não vira indicador.

## 05. Resumir grupos
### groupby() e mean()
O resultado tem uma linha por serviço, não por paciente.

In [68]:
df.groupby("servico")["tempo_internacao_dias"].mean()

,idade
servico,
Cardiologia,68.266667
Clínica Médica,59.066667
Neurologia,57.800000
Ortopedia,57.200000


### size() e count()
`size()` conta linhas; `count()` conta só valores não ausentes.

In [71]:
df.groupby("servico").size()

,0
servico,
Cardiologia,15
Clínica Médica,15
Neurologia,15
Ortopedia,15


In [72]:
print(df.groupby("servico")["tempo_internacao_dias"].count())

servico
Cardiologia       15
Clínica Médica    13
Neurologia        15
Ortopedia         15
Name: tempo_internacao_dias, dtype: int64


### agg(): várias medidas em uma tabela
A média de uma variável 0/1 é a proporção de valores iguais a 1.

In [73]:
indicadores = df.groupby("servico").agg(
    n_pacientes=("id_paciente", "size"),
    tempo_medio=("tempo_internacao_dias", "mean"),
    tempo_mediano=("tempo_internacao_dias", "median"),
    readmissao_prop=("readmissao_30d", "mean")
).reset_index()

indicadores.round(2)

,servico,n_pacientes,tempo_medio,tempo_mediano,readmissao_prop
0,Cardiologia,15,8.67,7.0,0.20
1,Clínica Médica,15,6.85,8.0,0.13
2,Neurologia,15,7.73,7.0,0.20
3,Ortopedia,15,3.80,3.0,0.27


> Um resumo facilita comparar grupos, mas esconde a variação entre os pacientes de cada grupo.

## 06. Prática: calcular e conferir
1. Crie `tempo_internacao_horas` sem laço e confira com `np.allclose()`.
2. Crie `teve_internacao_previa` com `np.where()`, usando `internacoes_anteriores > 0`.
3. Use `agg()` para obter número de pacientes e tempo médio por serviço.

In [36]:
# 1.

In [37]:
# 2.

In [38]:
# 3.